# Risk analysis and segmentation

In [1]:
import pandas as pd

## General risk profile

In [3]:
df_clean = pd.read_csv("C:/Users/Felhasználó/Documents/data analysis/credit-risk-analysis/risk analysis/data/credit_risk_dataset_cleaned.csv")

In [4]:
n_loans = len(df_clean)
status_counts = df_clean["loan_status"].value_counts().sort_index()
default_rate = df_clean["loan_status"].mean()

print(f"Total number of loans: {n_loans:,}")
print(f"Non-default loans: {status_counts.get(0, 0):,}")
print(f"Default loans: {status_counts.get(1, 0):,}")
print(f"Default rate: {default_rate:.2%}")

Total number of loans: 32,409
Non-default loans: 25,321
Default loans: 7,088
Default rate: 21.87%


C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_13980\652340852.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Non-default loans: {status_counts.get(0, 0):,}")
C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_13980\652340852.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Default loans: {status_counts.get(1, 0):,}")


Out of all the records, 6,939 observations are default loans taking up 21.91% of the cleaned dataset, suggesting a moderate imbalance between default and non-default representations. What if this imbalance is also present in different categories, suggesting that some factors are associated with higher risk than others?

The next part of the notebook will be investigating, if there are categories that contribute to higher default rates in the data.

In [6]:
def q_bucketer(data):
    return pd.cut(data, bins=4, labels=["Q1", "Q2", "Q3", "Q4"])

def bucketer(data):
    return pd.cut(data, bins=3, labels=["low", "medium", "high"])

df_clean.groupby(q_bucketer(df_clean["loan_pct_income"]))["loan_status"].mean()
df_clean.groupby(bucketer(df_clean["loan_pct_income"]))["loan_status"].mean()

C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_13980\1083881233.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_clean.groupby(q_bucketer(df_clean["loan_pct_income"]))["loan_status"].mean()
C:\Users\Felhasználó\AppData\Local\Temp\ipykernel_13980\1083881233.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_clean.groupby(bucketer(df_clean["loan_pct_income"]))["loan_status"].mean()


loan_pct_income
low       0.148381
medium    0.584831
high      0.767857
Name: loan_status, dtype: float64

## Risk analysis

In [6]:
def risk_by(column):
    return df_clean.groupby(column).agg(
        loan_count=("loan_status", "size"),
        default_count=("loan_status", "sum"),
        default_rate=("loan_status", "mean")
    )

risk_by_grade = risk_by("loan_grade")
display(risk_by_grade)

,loan_count,default_count,default_rate
loan_grade,,,
A,10702,1066,0.099608
B,10384,1695,0.163232
C,6436,1336,0.207582
D,3619,2137,0.590495
E,963,621,0.644860
F,241,170,0.705394
G,64,63,0.984375


Since loan grades represent the likelihood of a borrower defaulting, it would be logical to assume that higher default rates will be observed in grades associated with higher risk (letters placed later in the alphabet). The output above seems to confirm this theory as there is a consistent increase in the default rate towards higher risk categories, however, it is noteable that these groups are also less represented in the data, raising a bit of uncertainty about the verification of the assumption.

In [ ]:
for col in ["loan_intent", "person_home_ownership", "cb_person_default_on_file"]:
    risk_table = risk_by(col)
    display(risk_table)

The results show that higher default rates occur more in groups with the loan purpose of debt consolidation, home improvement and settling medical costs. Also, the default rate is significantly higher in groups with rentals or with the "other" home ownership category, however, since there are only 105 records observed in the latter, we cannot be certain that the default rate would not change with more data at hand. While the distibution of records is also skewed within the categories of the cb_person_default_on_file column, there is an observed increase in default rates of people with a history of defaulting than those without.

Now, we will be investigating, if other factors like interest rate or loan burden differentiate the data as well. 

In [ ]:
bins = [0, 0.10, 0.20, 0.30, 0.40, 0.50, 1.00]

df_clean["loan_income_band"] = pd.cut(
    df_clean["loan_percent_income"], bins=bins, right=True, include_lowest=True)

burden_analysis = (df_clean.groupby("loan_income_band", observed=False)
    .agg(
        loan_count=("loan_status", "size"),
        default_count=("loan_status", "sum"),
        default_rate=("loan_status", "mean")
    )
)
display(burden_analysis)

BOXPLOT:

In [ ]:
data = df_clean[["loan_grade", "loan_int_rate"]].dropna()
order = sorted(data["loan_grade"].unique())

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=data, x="loan_grade", y="loan_int_rate", order=order, color="mediumturquoise", ax=ax)
ax.set_title("Interest rate by loan grade")
plt.show()

DELETE:

In [4]:
numeric_cols = df_clean.select_dtypes(include="number")
categorical_cols = df_clean.select_dtypes(include="object")
binary_cols = df_clean.select_dtypes(include="bool")
cat_bool_cols = categorical_cols.columns.append(binary_cols.columns)

In [7]:
Q1 = numeric_cols.quantile(.25, axis = 0)
Q3 = numeric_cols.quantile(.75, axis = 0)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers_list = []

for col in numeric_cols.columns:
    for idx, value in numeric_cols[col].items():
        if value < lower_limit[col] or value > upper_limit[col]:
            outliers_list.append({"row": idx, "column": col, "value": value})

outliers_iqr = pd.DataFrame(outliers_list).set_index("row")

outlier_summary = outliers_iqr.groupby("column").agg(
    count=("value", "count"),
    min_outlier=("value", "min"),
    max_outlier=("value", "max")
)

outliers_pct_list = []

for col in numeric_cols.columns:
    pct = outlier_summary.loc[col]["count"] / numeric_cols[col].count() * 100
    outliers_pct_list.append({"idx": col, "pct": pct, "upper_limit": upper_limit[col]})

outliers_pct = pd.DataFrame(outliers_pct_list).set_index("idx")
outlier_summary = pd.concat([outlier_summary, outliers_pct], axis = 1)

display(outlier_summary)
print(f"Total number of outliers: {outliers_iqr.index.nunique()}")
print(f"Total percentage of outliers: {outliers_iqr.index.nunique() / len(numeric_cols) * 100:.2f}%")

,count,min_outlier,max_outlier,pct,upper_limit
cb_person_cred_hist_length,1138,16.000000,30.00,3.511370,15.500000
loan_amnt,1678,23200.000000,35000.00,5.177574,23125.000000
loan_int_rate,6,22.060000,23.22,0.020467,21.825000
loan_pct_income,732,0.438356,0.83,2.258632,0.438333
person_age,1486,41.000000,94.00,4.585146,40.500000
person_emp_length,850,15.000000,41.00,2.696529,14.500000
person_income,1474,140300.000000,2039784.00,4.548119,140250.000000


Total number of outliers: 5576
Total percentage of outliers: 17.21%
